In [2]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

In [3]:
from data.get_data import get_dataframe
from data_processor.calculate_stats import calculate_statistics
from data_processor.data_categorising import categories_columns
from data_processor.data_cleaner import clean_data
from data_processor.fight_stats import finalProcessingForFighter, calculateAverages
from data_processor.data_types_fixes import check_and_process_data_type, drop_col_for_training
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
import numpy as np

In [4]:
og_df = get_dataframe('original.csv')
cleaned_df = clean_data(og_df)
cleaned_df["target"] = (cleaned_df["Winner"] == cleaned_df["Fighter 1"]).astype(int)

stats_df = calculate_statistics(cleaned_df)
processed_df = finalProcessingForFighter(stats_df)
processed_df = check_and_process_data_type(processed_df)
avg_df = calculateAverages(processed_df)
cat_df = categories_columns(avg_df)
df_for_training = drop_col_for_training(cat_df)

In [5]:
X = df_for_training.drop(columns=["Target"])
y = df_for_training["Target"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])


In [8]:
param_grid = {
    "knn__n_neighbors": range(1, 51),
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2], 
    "knn__leaf_size": [10, 20, 30, 40, 50],
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="accuracy",
    n_jobs=-1
)

In [9]:
grid.fit(X_train, y_train)

GridSearchCV(estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('knn', KNeighborsClassifier())]),
             n_jobs=-1,
             param_grid={'knn__leaf_size': [10, 20, 30, 40, 50],
                         'knn__n_neighbors': range(1, 51), 'knn__p': [1, 2],
                         'knn__weights': ['uniform', 'distance']},
             scoring='accuracy')

In [10]:
y_pred = grid.predict(X_test)
y_proba = grid.predict_proba(X_test)[:, 1]

In [11]:
print("Best CV accuracy:", grid.best_score_)
print("Best parameters:", grid.best_params_)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_proba))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Best CV accuracy: 0.5328085432236966
Best parameters: {'knn__leaf_size': 10, 'knn__n_neighbors': 42, 'knn__p': 1, 'knn__weights': 'uniform'}

Test Accuracy: 0.5299695225194717
ROC-AUC Score: 0.5463864951691775

Confusion Matrix:
 [[878 621]
 [767 687]]

Classification Report:
               precision    recall  f1-score   support

           0       0.53      0.59      0.56      1499
           1       0.53      0.47      0.50      1454

    accuracy                           0.53      2953
   macro avg       0.53      0.53      0.53      2953
weighted avg       0.53      0.53      0.53      2953

